# Prometheus v7.1 — Draft Composition Analysis

This notebook provides an exploratory analysis of Dota 2 hero draft compositions using the Prometheus AI engine.

## Goals
- Analyze hero attribute distribution in the database
- Visualize draft composition scores
- Identify optimal team compositions by role
- Benchmark hero ratings across key metrics

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from collections import Counter, defaultdict
from src.draft_analyzer import HERO_PROFILES, Role, Attribute, Timing, HeroProfile
from src.hero_mapper import HERO_DATA

print(f'✅ Loaded {len(HERO_PROFILES)} hero profiles from draft analyzer')
print(f'✅ Loaded {len(HERO_DATA)} heroes from hero mapper')

## 1. Hero Database Overview

In [ ]:
# Attribute distribution
attr_counts = Counter(h.attribute.value for h in HERO_PROFILES.values())
print('=== Attribute Distribution ===')
for attr, count in sorted(attr_counts.items(), key=lambda x: -x[1]):
    bar = '█' * count
    print(f'  {attr:12s} {bar} ({count})')

# Role distribution
role_counts = Counter()
for h in HERO_PROFILES.values():
    for r in h.roles:
        role_counts[r.value] += 1

print('\n=== Role Distribution ===')
for role, count in sorted(role_counts.items(), key=lambda x: -x[1]):
    bar = '█' * count
    print(f'  {role:15s} {bar} ({count})')

# Timing distribution
timing_counts = Counter(h.timing.value for h in HERO_PROFILES.values())
print('\n=== Timing Distribution ===')
for timing, count in sorted(timing_counts.items(), key=lambda x: -x[1]):
    bar = '█' * count
    print(f'  {timing:8s} {bar} ({count})')

## 2. Top Heroes by Key Metrics

In [ ]:
def top_heroes_by_metric(metric: str, n: int = 5) -> list:
    """Return top N heroes sorted by a given metric."""
    return sorted(
        HERO_PROFILES.values(),
        key=lambda h: getattr(h, metric),
        reverse=True
    )[:n]


metrics = ['teamfight', 'mobility', 'control', 'push', 'pickoff']

for metric in metrics:
    top = top_heroes_by_metric(metric, n=5)
    print(f'\n🏆 Top 5 by {metric.upper()}:')
    for i, h in enumerate(top, 1):
        score = getattr(h, metric)
        bar = '█' * int(score)
        print(f'  {i}. {h.name:25s} {bar} {score:.1f}')

## 3. Draft Composition Scorer

In [ ]:
def score_draft(hero_names: list) -> dict:
    """Score a 5-hero draft across multiple dimensions."""
    heroes = [HERO_PROFILES[n] for n in hero_names if n in HERO_PROFILES]
    
    if not heroes:
        return {}
    
    n = len(heroes)
    scores = {
        'teamfight': sum(h.teamfight for h in heroes) / n,
        'push':      sum(h.push for h in heroes) / n,
        'pickoff':   sum(h.pickoff for h in heroes) / n,
        'mobility':  sum(h.mobility for h in heroes) / n,
        'control':   sum(h.control for h in heroes) / n,
        'save':      sum(h.save for h in heroes) / n,
    }
    
    # Timing profile
    timings = Counter(h.timing.value for h in heroes)
    scores['timing_profile'] = dict(timings)
    
    # Damage type
    damage_types = Counter(h.damage_type for h in heroes)
    scores['damage_types'] = dict(damage_types)
    
    # Overall score (weighted)
    scores['overall'] = (
        scores['teamfight'] * 0.30 +
        scores['push'] * 0.20 +
        scores['control'] * 0.20 +
        scores['pickoff'] * 0.15 +
        scores['mobility'] * 0.15
    )
    
    return scores


# Compare two draft strategies
teamfight_draft = ['Faceless Void', 'Invoker', 'Mars', 'Enigma', 'Dazzle']
split_push_draft = ['Terrorblade', 'Tinker', 'Nature\'s Prophet', 'Lycan', 'Beastmaster']
balanced_draft = ['Juggernaut', 'Invoker', 'Axe', 'Lion', 'Dazzle']

drafts = {
    'Teamfight': teamfight_draft,
    'Split Push': split_push_draft,
    'Balanced': balanced_draft,
}

print('=== Draft Comparison ===')
for draft_name, heroes in drafts.items():
    available = [h for h in heroes if h in HERO_PROFILES]
    if len(available) < 3:
        print(f'\n{draft_name}: insufficient heroes in database')
        continue
    scores = score_draft(available)
    print(f'\n📋 {draft_name} Draft ({len(available)} heroes):')
    print(f'   Heroes: {", ".join(available)}')
    for metric in ['teamfight', 'push', 'control', 'pickoff', 'mobility']:
        val = scores.get(metric, 0)
        bar = '█' * int(val)
        print(f'   {metric:12s} {bar} {val:.2f}')
    print(f'   Overall Score: {scores.get("overall", 0):.2f}/10')

## 4. Hero Correlation Matrix

Which heroes have similar stat profiles? This can help identify synergies and counters.

In [ ]:
import math

METRICS = ['teamfight', 'push', 'pickoff', 'mobility', 'control', 'save', 'lane_presence']

def hero_vector(hero: HeroProfile) -> list:
    return [getattr(hero, m) for m in METRICS]

def cosine_sim(a: list, b: list) -> float:
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x**2 for x in a))
    nb = math.sqrt(sum(x**2 for x in b))
    return dot / (na * nb) if na and nb else 0.0

def find_similar_heroes(hero_name: str, n: int = 5) -> list:
    if hero_name not in HERO_PROFILES:
        return []
    target = hero_vector(HERO_PROFILES[hero_name])
    sims = []
    for name, hero in HERO_PROFILES.items():
        if name == hero_name:
            continue
        sim = cosine_sim(target, hero_vector(hero))
        sims.append((name, sim))
    return sorted(sims, key=lambda x: -x[1])[:n]


# Find heroes similar to key picks
for hero_name in ['Invoker', 'Faceless Void', 'Lion']:
    similar = find_similar_heroes(hero_name, n=4)
    if similar:
        print(f'\nHeroes similar to {hero_name}:')
        for name, sim in similar:
            print(f'  {name:25s} similarity: {sim:.4f}')